# Generación de imágenes sintéticas de mamografía con GAN y momentos de Hu

- Proyecto de aumento de datos para imagenología médica: una GAN genera imágenes sintéticas de mamografía para balancear clases minoritarias.
- Contribución propia: los **momentos de Hu** como métrica morfológica complementaria a FID/KID, específica para imágenes que pueden binarizarse con sentido clínico.
- Artículo completo, con el diagrama del pipeline y las decisiones de ingeniería explicadas: https://fuzzyfrog.ai/es/ai-lab/proyectos/salud/gan-imagenes-sinteticas-mamografia-momentos-hu/
- **Nota:** este notebook usa el dataset público CBIS-DDSM. No contiene datos ni identificadores de ningún paciente o institución privada.


## Diagrama de la arquitectura

- Dos redes en competencia: **Generador** (ruido → imagen sintética 256×256×3) y **Discriminador** (imagen real o sintética → probabilidad de "real").
- El discriminador se congela (`trainable = False`) mientras se entrena el generador dentro del modelo combinado.
- Una vez entrenada la GAN, las imágenes sintéticas se evalúan (FID, KID, momentos de Hu) y se usan para entrenar un clasificador CNN downstream.
- Diagrama editable disponible en el artículo de la plataforma (liga arriba).


## Carga de datos

- Dataset público: [CBIS-DDSM Breast Cancer Image Dataset](https://www.kaggle.com/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset) en Kaggle.
- Combina DDSM (casos negativos) y CBIS-DDSM (casos positivos), normalizado a 299×299 px en este resumen técnico.
- Para esta demostración se trabaja con una muestra reducida, suficiente para ilustrar el pipeline sin requerir el volumen completo.


In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras import layers, Model

# Descarga vía kagglehub (requiere credenciales de Kaggle configuradas)
# import kagglehub
# dataset_path = kagglehub.dataset_download("awsaf49/cbis-ddsm-breast-cancer-image-dataset")

DATASET_PATH = "data/cbis-ddsm"  # ajustar a la ruta local tras la descarga
IMG_SIZE = 256

def load_image(path, size=IMG_SIZE):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (size, size))
    img = img.astype("float32") / 127.5 - 1.0  # normalizado a [-1, 1] para el generador
    return img

# metadata: id_imagen, ruta, clase (0=negativo ... 4=masa maligna)
metadata = pd.read_csv(os.path.join(DATASET_PATH, "metadata.csv"))
print(metadata["clase"].value_counts())


## Explicación de datos

- Etiquetado dual: binario (0 = negativo, 1 = positivo) y multiclase (0 negativo, 1 calcificación benigna, 2 masa benigna, 3 calcificación maligna, 4 masa maligna).
- El dataset real combinado tiene un fuerte desbalance: 14% positivos frente a 86% negativos.
- Las positivas (CBIS-DDSM) se recortan por región de interés vía máscara, con relleno de contexto y variaciones aleatorias de recorte/rotación antes de redimensionar.


In [ ]:
print(f"Total de ejemplos: {len(metadata)}")
print(f"Proporción positivos: {(metadata['clase'] != 0).mean():.2%}")

muestra = metadata.groupby("clase").sample(n=3, random_state=42)
for _, fila in muestra.iterrows():
    img = load_image(fila["ruta"])
    print(fila["clase"], img.shape, img.min(), img.max())


## Análisis de datos / EDA

- Antes de generar nada sintético, conviene ver cómo se comportan las imágenes reales por clase.
- Aquí se calculan los momentos de Hu sobre las imágenes reales, que luego servirán como línea base para comparar contra las generadas.


In [ ]:
def momentos_hu(img):
    gris = cv2.cvtColor(((img + 1.0) * 127.5).astype("uint8"), cv2.COLOR_BGR2GRAY)
    _, binaria = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    momentos = cv2.moments(binaria)
    hu = cv2.HuMoments(momentos).flatten()
    # transformación logarítmica para poner los momentos en rangos comparables
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)
    return hu_log

hu_reales = np.array([momentos_hu(load_image(r["ruta"])) for _, r in muestra.iterrows()])
pd.DataFrame(hu_reales, columns=[f"Hu_{i+1}" for i in range(7)])


## Modelado

- **Generador:** `Dense(4096) → LeakyReLU → Reshape(4,4,256) → 6× [Conv2DTranspose → LeakyReLU]` upsampleando 8→16→32→64→128→256, `Conv2D` final a 3 canales.
- **Discriminador:** 7 bloques `Conv2D → LeakyReLU` (canales 64→64→128→128→128→256→256) → `Flatten → Dropout → Dense(1, sigmoide)`.
- Decisión de ingeniería: `BatchNormalization` se probó en el generador y se descartó por no mejorar resultados. Un único punto de `Dropout` (antes de la densa final) rindió igual que aplicarlo en cada bloque convolucional.
- Al entrenar el modelo combinado, el discriminador se congela (`trainable = False`) para que solo se actualicen los pesos del generador.


In [ ]:
LATENT_DIM = 100

def construir_generador():
    entrada = layers.Input(shape=(LATENT_DIM,))
    x = layers.Dense(4 * 4 * 256)(entrada)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Reshape((4, 4, 256))(x)
    for _ in range(6):  # 4 -> 8 -> 16 -> 32 -> 64 -> 128 -> 256
        x = layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
    salida = layers.Conv2D(3, kernel_size=4, padding="same", activation="tanh")(x)
    return Model(entrada, salida, name="generador")

def construir_discriminador():
    entrada = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    canales = [64, 64, 128, 128, 128, 256, 256]
    x = entrada
    for i, c in enumerate(canales):
        strides = 2 if i % 2 == 0 else 1
        x = layers.Conv2D(c, kernel_size=4, strides=strides, padding="same")(x)
        x = layers.LeakyReLU(0.2)(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.3)(x)
    salida = layers.Dense(1, activation="sigmoid")(x)
    return Model(entrada, salida, name="discriminador")

generador = construir_generador()
discriminador = construir_discriminador()
discriminador.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Modelo combinado: solo entrena al generador
discriminador.trainable = False
entrada_gan = layers.Input(shape=(LATENT_DIM,))
gan_salida = discriminador(generador(entrada_gan))
gan = Model(entrada_gan, gan_salida, name="gan_combinada")
gan.compile(optimizer="adam", loss="binary_crossentropy")

generador.summary()
discriminador.summary()


## Evaluación

- **FID / KID:** distancia entre activaciones de una capa de Inception v3 para imágenes reales vs. generadas, por clase.
- **Momentos de Hu:** comparación de histogramas reales vs. generados con correlación, chi-cuadrada, intersección y distancia de Bhattacharyya.
- **Downstream:** una CNN clasificadora entrenada con datos reales + sintéticos, comparada contra entrenar solo con datos reales y contra aumento tradicional (`ImageDataGenerator`).


In [ ]:
from scipy.linalg import sqrtm
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input

inception = InceptionV3(include_top=False, pooling="avg", input_shape=(299, 299, 3))

def activaciones_inception(imagenes):
    imagenes = tf.image.resize(imagenes, (299, 299))
    imagenes = preprocess_input((imagenes + 1.0) * 127.5)
    return inception.predict(imagenes, verbose=0)

def fid(act_reales, act_generadas):
    mu1, mu2 = act_reales.mean(axis=0), act_generadas.mean(axis=0)
    sigma1 = np.cov(act_reales, rowvar=False)
    sigma2 = np.cov(act_generadas, rowvar=False)
    diff = mu1 - mu2
    covmean = sqrtm(sigma1.dot(sigma2))
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return diff.dot(diff) + np.trace(sigma1 + sigma2 - 2 * covmean)

def comparar_hu(hu_reales, hu_generadas):
    hist_r, _ = np.histogram(hu_reales, bins=20)
    hist_g, _ = np.histogram(hu_generadas, bins=20)
    hist_r = hist_r.astype("float32"); hist_g = hist_g.astype("float32")
    return {
        "correlacion": cv2.compareHist(hist_r, hist_g, cv2.HISTCMP_CORREL),
        "chi_cuadrada": cv2.compareHist(hist_r, hist_g, cv2.HISTCMP_CHISQR),
        "interseccion": cv2.compareHist(hist_r, hist_g, cv2.HISTCMP_INTERSECT),
        "bhattacharyya": cv2.compareHist(hist_r, hist_g, cv2.HISTCMP_BHATTACHARYYA),
    }

# Resultados de referencia reportados en el proyecto original (ilustrativos en este notebook):
resultados_fid_kid = pd.DataFrame({
    "clase": ["negativo", "calcificacion_benigna", "masa_benigna", "calcificacion_maligna", "masa_maligna"],
    "KID": [0.09, 0.13, 0.15, 0.19, 0.236],
    "FID": [118.4, 142.7, 156.9, 189.3, 218.2],
})
resultados_fid_kid


In [ ]:
# Comparación downstream: CNN Vanilla vs. Xception, real+GAN vs. real vs. aumento tradicional
resultados_downstream = pd.DataFrame({
    "arquitectura": ["CNN Vanilla", "CNN Vanilla", "CNN Vanilla", "Xception", "Xception", "Xception"],
    "configuracion": ["real+GAN", "solo real", "real+aumento tradicional"] * 2,
    "accuracy": [0.87, 0.83, 0.64, 0.70, 0.83, 0.64],
})
resultados_downstream.pivot(index="configuracion", columns="arquitectura", values="accuracy")


## Hallazgos principales

- El aumento de datos con GAN superó consistentemente al aumento tradicional (`ImageDataGenerator`), que fue la configuración con peor desempeño en ambas arquitecturas probadas (0.64 de exactitud).
- Con una CNN sencilla, combinar datos reales y sintéticos (0.87) superó tanto a no aumentar (0.83) como al aumento tradicional. Con Xception y transferencia de aprendizaje, el patrón se invirtió parcialmente: entrenar solo con datos reales (0.83) superó a la mezcla con sintéticos (0.70).
- La clase "masa maligna" fue la más difícil de replicar para la GAN, con los valores de FID y KID más altos entre las cinco clases.
- Los momentos de Hu, propuestos como métrica propia para este caso, mostraron distribuciones visualmente similares entre imágenes reales y generadas para las cinco clases, respaldando cualitativamente la calidad de la generación.
- Conclusión de fondo: la GAN no es una mejora garantizada en todos los escenarios, pero sí es una alternativa consistentemente mejor que el aumento de datos tradicional para este dominio.
